# Portion 1 — Data audit, preprocessing & leakage control (CPU)
DNA-origami HS-AFM dataset (Zenodo 14275812). Runs on a **CPU runtime** — no GPU needed.

**Before you start:** put these on Google Drive
- the Zenodo zips (`2nM.zip … 12nM.zip`, `overview4x4um.zip`) in one folder, e.g. `MyDrive/dna_origami/raw/`
- `portion1_pipeline.py` in e.g. `MyDrive/dna_origami/code/`

Run the cells top to bottom. Cell 4 is a 2-minute test; cell 5 is the full run (~1–2 h on Colab CPU).

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Paths - EDIT THESE if your folders are different
RAW_DIR  = '/content/drive/MyDrive/dna_origami/raw'                 # folder with the zips
CODE     = '/content/drive/MyDrive/dna_origami/code/portion1_pipeline.py'
OUT_DRIVE = '/content/drive/MyDrive/dna_origami/portion1_output'     # final results on Drive

# Work on the fast local disk, copy results to Drive at the end.
COPY_ZIPS_LOCALLY = True       # ~11 GB copy, makes reading ~5-10x faster than from Drive
LOCAL_RAW = '/content/raw'
LOCAL_OUT = '/content/portion1_output'

import os, glob
assert os.path.exists(CODE), f'Script not found: {CODE}'
zips = sorted(glob.glob(os.path.join(RAW_DIR, '*.zip')))
print('Found archives:'); [print('  ', os.path.basename(z), round(os.path.getsize(z)/1e9, 2), 'GB') for z in zips]
assert zips or os.listdir(RAW_DIR), 'No data found in RAW_DIR'

In [ ]:
# 3. Install / check dependencies (Colab already has most of them)
!pip -q install tifffile scikit-image tabulate
import shutil, subprocess
shutil.copy(CODE, '/content/portion1_pipeline.py')
if COPY_ZIPS_LOCALLY:
    os.makedirs(LOCAL_RAW, exist_ok=True)
    for z in zips:
        dst = os.path.join(LOCAL_RAW, os.path.basename(z))
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(z):
            print('copying', os.path.basename(z)); shutil.copy(z, dst)
    DATA = LOCAL_RAW
else:
    DATA = RAW_DIR
print('Reading data from', DATA, '| CPUs:', os.cpu_count())

In [ ]:
# 4. QUICK TEST (first 15 frames of each session, ~2 min). Must finish without errors.
!python /content/portion1_pipeline.py --data "$DATA" --out /content/p1_quicktest --max-frames 15 --skip-overview --steps index preprocess qc splits audit figures report
print(open('/content/p1_quicktest/summary.md').read())

In [ ]:
# 5. FULL RUN (all steps). Safe to re-run single steps later with --steps ...
!python /content/portion1_pipeline.py --data "$DATA" --out "$LOCAL_OUT"

In [ ]:
# 6. Look at the results
from IPython.display import Image, display, Markdown
display(Markdown(open(f'{LOCAL_OUT}/summary.md').read()))
for f in sorted(glob.glob(f'{LOCAL_OUT}/figures/*.png')):
    print(os.path.basename(f)); display(Image(f, width=750))

In [ ]:
# 7. Copy everything to Google Drive (stacks are ~4-6 GB; use --no-raw in cell 5 to halve)
os.makedirs(OUT_DRIVE, exist_ok=True)
!rsync -a --info=progress2 "$LOCAL_OUT"/ "$OUT_DRIVE"/
print('Saved to', OUT_DRIVE)